# Experiment: SpectralBio Final Accept Hardening Part 6 (L4)

This notebook is the breadth-and-failure-analysis notebook.

## Scope
- Expand the support-ranked panel from 10 genes to top 25 feasible genes
- Score the panel under 150M and, if feasible, 650M shadow mode
- Run BRCA1 full-set failure analysis by domain, review status, substitution class, and confidence strata
- Generate tables and figures that convert BRCA1 from a naked weakness into explained heterogeneity

## Intended runtime
- Target hardware: L4
- Optional A100 rerun if we decide to push 650M across the full top-25 panel


In [ ]:
from pathlib import Path

USE_GOOGLE_DRIVE = False
DRIVE_MOUNT_POINT = Path('/content/drive')
DRIVE_OUTPUT_SUBDIR = Path('MyDrive/SpectralBioFinalAcceptA100')

if USE_GOOGLE_DRIVE:
    from google.colab import drive
    drive.mount(str(DRIVE_MOUNT_POINT))
    print('Drive mounted at', DRIVE_MOUNT_POINT)
else:
    print('Drive mount skipped; outputs stay in the VM unless OUTPUT_ROOT is changed later.')


In [ ]:
import os
import subprocess
import sys
from pathlib import Path

REPO_URL = 'https://github.com/DaviBonetto/SpectralBio.git'
REPO_BRANCH = 'codex/claw4s-rebuild'
DEFAULT_REPO_ROOT = Path('/content/Stanford-Claw4s')
ENV_REPO_ROOT = os.environ.get('SPECTRALBIO_REPO_ROOT', '').strip()


def _looks_like_repo(path: Path) -> bool:
    return (path / 'src' / 'spectralbio').exists() and (path / 'notebooks').exists()


candidate_roots = []
if ENV_REPO_ROOT:
    candidate_roots.append(Path(ENV_REPO_ROOT).expanduser())
candidate_roots.extend([Path.cwd(), Path.cwd().parent, DEFAULT_REPO_ROOT])

REPO_ROOT = next((path.resolve() for path in candidate_roots if _looks_like_repo(path)), DEFAULT_REPO_ROOT)
BOOTSTRAP_MARKER = REPO_ROOT / '.colab_bootstrap_v4_complete'

if not _looks_like_repo(REPO_ROOT):
    REPO_ROOT.parent.mkdir(parents=True, exist_ok=True)
    subprocess.check_call(['git', 'clone', '--branch', REPO_BRANCH, '--single-branch', REPO_URL, str(REPO_ROOT)])

os.chdir(REPO_ROOT)
subprocess.check_call(['git', 'fetch', 'origin', REPO_BRANCH])
subprocess.check_call(['git', 'checkout', REPO_BRANCH])
src_path = REPO_ROOT / 'src'
if str(src_path) not in sys.path:
    sys.path.insert(0, str(src_path))

if not BOOTSTRAP_MARKER.exists():
    subprocess.check_call([sys.executable, '-m', 'pip', 'uninstall', '-y', 'torchvision'])
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-e', '.', '--no-deps'])
    subprocess.check_call([
        sys.executable, '-m', 'pip', 'install',
        'numpy==2.1.3', 'plotly==5.24.1', 'pyyaml==6.0.2', 'scikit-learn==1.5.2',
        'scipy==1.14.1', 'transformers==4.49.0', 'accelerate>=1.0.0', 'pandas', 'matplotlib'
    ])
    BOOTSTRAP_MARKER.write_text('ok\n', encoding='utf-8')
    print('Dependencies installed. Continuing in the same runtime.')
else:
    print('Bootstrap marker found; skipping reinstall.')


## Plan

- Replace the impression of a narrow support-ranked panel with a visibly broader surface.
- Keep BRCA1 failure analysis principled and predeclared.
- Produce one table for breadth and one figure family for BRCA1 heterogeneity.


In [ ]:
import shutil
import subprocess
import sys
import time
from pathlib import Path

import pandas as pd
from IPython.display import FileLink, HTML, Javascript, display

try:
    from spectralbio.supplementary.final_accept_hardening import (
        AcceptHardeningConfig,
        create_accept_hardening_paths,
        run_brca1_failure_analysis,
        run_support_ranked_panel_expansion,
    )
except ImportError:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-e', '.', '--no-deps'])
    from spectralbio.supplementary.final_accept_hardening import (
        AcceptHardeningConfig,
        create_accept_hardening_paths,
        run_brca1_failure_analysis,
        run_support_ranked_panel_expansion,
    )

TOP_K_GENES = 25
RUN_650M_SHADOW = True
BRCA1_STRATA = ('domain', 'review_status', 'substitution_class', 'confidence')
OVERWRITE = False
DOWNLOAD_ZIP_NAME = 'final_accept_part6_panel25_brca1_failure_results.zip'

OUTPUT_ROOT = (
    DRIVE_MOUNT_POINT / DRIVE_OUTPUT_SUBDIR
    if USE_GOOGLE_DRIVE
    else REPO_ROOT / 'supplementary' / 'final_accept_a100'
)

config = AcceptHardeningConfig(
    stronger_model_names=('facebook/esm2_t33_650M_UR50D', 'facebook/esm2_t36_3B_UR50D'),
    max_additional_genes=23,
    overwrite=OVERWRITE,
)
paths = create_accept_hardening_paths(repo_root=REPO_ROOT, output_root=OUTPUT_ROOT)
print(paths)
print('Top-K genes:', TOP_K_GENES)
print('BRCA1 strata:', BRCA1_STRATA)



In [ ]:
if 'run_support_ranked_panel_expansion' not in globals() or 'run_brca1_failure_analysis' not in globals():
    from spectralbio.supplementary.final_accept_hardening import (
        run_brca1_failure_analysis,
        run_support_ranked_panel_expansion,
    )

print('Starting top-25 support-ranked panel expansion...')
panel_start = time.time()
panel25_summary = run_support_ranked_panel_expansion(
    paths=paths,
    config=config,
    top_k=TOP_K_GENES,
    run_650m_shadow=RUN_650M_SHADOW,
)
print(f'Panel expansion finished in {(time.time() - panel_start) / 60:.1f} min')

print('Starting BRCA1 failure analysis...')
fail_start = time.time()
brca1_failure = run_brca1_failure_analysis(
    paths=paths,
    config=config,
    strata=BRCA1_STRATA,
)
print(f'BRCA1 failure analysis finished in {(time.time() - fail_start) / 60:.1f} min')

display(pd.DataFrame(panel25_summary['rows']).head())
display(pd.DataFrame(brca1_failure['rows']).head())
print('Results root:', paths.root)



In [ ]:
part6_bundle_root = paths.root / '_downloads' / 'part6_panel25_brca1_failure'
if part6_bundle_root.exists():
    shutil.rmtree(part6_bundle_root)
part6_bundle_root.mkdir(parents=True, exist_ok=True)

for folder_name in ('panel25', 'brca1_failure'):
    source = paths.root / folder_name
    if source.exists():
        shutil.copytree(source, part6_bundle_root / folder_name, dirs_exist_ok=True)

manifest_lines = [
    'Part 6 bundled outputs',
    f'Results root: {paths.root}',
    f'Generated at: {pd.Timestamp.utcnow()}',
    '',
    'Included folders:',
    '- panel25',
    '- brca1_failure',
]
(part6_bundle_root / 'bundle_manifest.txt').write_text('\n'.join(manifest_lines) + '\n', encoding='utf-8')

archive_path = REPO_ROOT / 'notebooks' / DOWNLOAD_ZIP_NAME
if archive_path.exists():
    archive_path.unlink()
archive_base = archive_path.with_suffix('')
shutil.make_archive(str(archive_base), 'zip', root_dir=part6_bundle_root.parent, base_dir=part6_bundle_root.name)
print('ZIP ready:', archive_path)
display(FileLink(str(archive_path), result_html_prefix='Download ZIP: '))

try:
    from google.colab import files
    files.download(str(archive_path))
except Exception:
    download_href_candidates = [
        f'/files/notebooks/{archive_path.name}',
        f'files/notebooks/{archive_path.name}',
        archive_path.name,
    ]
    links_html = ''.join(
        f'<li><a class="part6-download-link" href="{href}" download>{href}</a></li>'
        for href in download_href_candidates
    )
    display(HTML(
        '<div>'
        '<p>If automatic download is blocked, click one of the links below.</p>'
        f'<ul>{links_html}</ul>'
        '</div>'
    ))
    js = """
(() => {
  const links = Array.from(document.querySelectorAll('.part6-download-link'));
  const preferred = links.find((link) => link.getAttribute('href').startsWith('/files/')) || links[0];
  if (preferred) {
    preferred.click();
  }
})();
"""
    display(Javascript(js))


In [ ]:
print('Part 6 notebook complete.')
print('Output root:', paths.root)
if 'archive_path' in globals():
    print('ZIP archive:', archive_path)
else:
    print('Run the previous cell to build the zip archive.')

